                                  Stage 2  XGBoost Ranker Training for Recommendation System
The XGBoost ranker effectively learns to rank candidate items using behavioral, contextual, and embedding-based features. This enables the recommendation system to deliver highly relevant, personalized, and accurate item suggestions.
## Objective
To train a machine learning ranking model that predicts the relevance score of candidate items and ranks them accordingly for personalized recommendations.

## Data Source
The model uses the previously generated dataset:

ranker_training_dataset.csv → contains features, labels, and group IDs

## Data Loading and Inspection
The dataset is loaded into a DataFrame
Column names are cleaned
Data shape and sample rows are displayed
Purpose:
To verify dataset integrity before training.

## Data Validation

Ensures all required columns are present
Checks label distribution (positive vs negative samples)
Validates group IDs
Purpose:
To ensure the dataset is suitable for ranking model training.
## Data Cleaning and Preprocessing

Missing categorical values are replaced with “Unknown”
Numerical columns are converted to proper numeric types
Rows with invalid values are removed

## Feature Selection
Selected features include:

basket_size
item_cooccurrence_score
category_affinity_score
context_popularity_score
customer_history_score
embedding_similarity_score
isHoliday, isFestival, weekOfMonth
season, timeSlot, candidate_category

## Target:
label
Group:group_id
Purpose:
To define inputs for the ranking model.

## Feature Encoding
Categorical features are converted using one-hot encoding
Validation and test sets are aligned with training columns

## Data Sorting by Group
Data is sorted by group ID
Ensures that ranking model processes grouped data correctly

## Model Training (XGBoost Ranker)
The model is trained using:
Objective: rank
Learning rate, depth, and regularization parameters
LambdaRank strategy for pairwise ranking

## Sample Prediction Output
Displays ranked candidates for a sample group
Shows predicted scores and actual labels
## Benefits

Learns optimal ranking from multiple signals
Improves recommendation accuracy
Supports personalized and context-aware recommendations
Scalable and efficient for large datasets
Provides interpretable feature importance

## Recommendation Flow
Generate candidate items
Compute feature vectors
Apply trained XGBoost ranker
Sort items by predicted score
Recommend top-N items

In [1]:
!pip install xgboost


  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

In [3]:
DATA_DIR = Path(r"D:\recommendation_item_API\data")

RANKER_FILE = DATA_DIR / "ranker_training_dataset.csv"
MODEL_DIR = DATA_DIR / "model_outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Ranker file exists:", RANKER_FILE.exists())
print("Model dir:", MODEL_DIR)

Ranker file exists: True
Model dir: D:\recommendation_item_API\data\model_outputs


In [4]:
ranker_df = pd.read_csv(RANKER_FILE)

ranker_df.columns = [c.strip() for c in ranker_df.columns]

print("Shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

Shape: (449680, 16)
Columns:
['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,13348,Bakery-Bread,2,16.5,6.0,5.0,1.0,0.997975,Winter,0,0,Morning,1,0,1
2,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0,1
3,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
4,23561,13352,Protein-Egg,2,32.0,4.0,13.0,2.0,0.998886,Winter,0,0,Morning,1,0,1


In [5]:
ranker_df = pd.read_csv(RANKER_FILE)

ranker_df.columns = [c.strip() for c in ranker_df.columns]

print("Shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

Shape: (449680, 16)
Columns:
['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,13348,Bakery-Bread,2,16.5,6.0,5.0,1.0,0.997975,Winter,0,0,Morning,1,0,1
2,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0,1
3,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
4,23561,13352,Protein-Egg,2,32.0,4.0,13.0,2.0,0.998886,Winter,0,0,Morning,1,0,1


In [6]:
required_cols = [
    "group_id",
    "label",
    "candidate_item_id",
    "candidate_category",
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "season",
    "isHoliday",
    "isFestival",
    "timeSlot",
    "weekOfMonth"
]

missing_cols = [c for c in required_cols if c not in ranker_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("Unique groups:", ranker_df["group_id"].nunique())
print("Label distribution:")
print(ranker_df["label"].value_counts(dropna=False))

Unique groups: 40880
Label distribution:
label
0    408800
1     40880
Name: count, dtype: int64


In [7]:
ranker_df["candidate_category"] = ranker_df["candidate_category"].fillna("Unknown").astype(str)
ranker_df["season"] = ranker_df["season"].fillna("Unknown").astype(str)
ranker_df["timeSlot"] = ranker_df["timeSlot"].fillna("Unknown").astype(str)

numeric_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "label"
]

for col in numeric_cols:
    ranker_df[col] = pd.to_numeric(ranker_df[col], errors="coerce")

ranker_df = ranker_df.dropna(subset=numeric_cols).copy()

print("Shape after cleaning:", ranker_df.shape)
display(ranker_df.head())

Shape after cleaning: (449680, 16)


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,13348,Bakery-Bread,2,16.5,6.0,5.0,1.0,0.997975,Winter,0,0,Morning,1,0,1
2,23561,12625,Pantry-Oils,2,2.0,6.0,0.0,4.0,0.990649,Winter,0,0,Morning,1,0,1
3,23561,976,Pantry-Rice,2,20.0,3.0,2.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
4,23561,13352,Protein-Egg,2,32.0,4.0,13.0,2.0,0.998886,Winter,0,0,Morning,1,0,1


In [8]:
all_groups = ranker_df["group_id"].drop_duplicates().tolist()
random.seed(42)
random.shuffle(all_groups)

n_groups = len(all_groups)
train_end = int(n_groups * 0.80)
valid_end = int(n_groups * 0.90)

train_groups = set(all_groups[:train_end])
valid_groups = set(all_groups[train_end:valid_end])
test_groups  = set(all_groups[valid_end:])

train_df = ranker_df[ranker_df["group_id"].isin(train_groups)].copy()
valid_df = ranker_df[ranker_df["group_id"].isin(valid_groups)].copy()
test_df  = ranker_df[ranker_df["group_id"].isin(test_groups)].copy()

print("Train shape:", train_df.shape, "groups:", train_df["group_id"].nunique())
print("Valid shape:", valid_df.shape, "groups:", valid_df["group_id"].nunique())
print("Test shape :", test_df.shape,  "groups:", test_df["group_id"].nunique())

Train shape: (359744, 16) groups: 32704
Valid shape: (44968, 16) groups: 4088
Test shape : (44968, 16) groups: 4088


In [9]:
feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "season",
    "timeSlot",
    "candidate_category"
]

target_col = "label"
group_col = "group_id"

print("Feature columns:", feature_cols)

Feature columns: ['basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'isHoliday', 'isFestival', 'weekOfMonth', 'season', 'timeSlot', 'candidate_category']


In [10]:
X_train_raw = train_df[feature_cols].copy()
X_valid_raw = valid_df[feature_cols].copy()
X_test_raw  = test_df[feature_cols].copy()

X_train = pd.get_dummies(X_train_raw, columns=["season", "timeSlot", "candidate_category"])
X_valid = pd.get_dummies(X_valid_raw, columns=["season", "timeSlot", "candidate_category"])
X_test  = pd.get_dummies(X_test_raw,  columns=["season", "timeSlot", "candidate_category"])

# train columns অনুযায়ী align
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col].astype(int).values
y_valid = valid_df[target_col].astype(int).values
y_test  = test_df[target_col].astype(int).values

qid_train = train_df[group_col].values
qid_valid = valid_df[group_col].values
qid_test  = test_df[group_col].values

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)

X_train shape: (359744, 59)
X_valid shape: (44968, 59)
X_test shape : (44968, 59)


In [11]:
def sort_by_qid(X, y, qid):
    temp = X.copy()
    temp["_label"] = y
    temp["_qid"] = qid
    
    temp = temp.sort_values("_qid").reset_index(drop=True)
    
    y_sorted = temp["_label"].values
    qid_sorted = temp["_qid"].values
    X_sorted = temp.drop(columns=["_label", "_qid"])
    
    return X_sorted, y_sorted, qid_sorted

X_train_sorted, y_train_sorted, qid_train_sorted = sort_by_qid(X_train, y_train, qid_train)
X_valid_sorted, y_valid_sorted, qid_valid_sorted = sort_by_qid(X_valid, y_valid, qid_valid)
X_test_sorted, y_test_sorted, qid_test_sorted = sort_by_qid(X_test, y_test, qid_test)

print("Sorted train shape:", X_train_sorted.shape)

Sorted train shape: (359744, 59)


In [12]:
ranker = xgb.XGBRanker(
    objective="rank:ndcg",
    tree_method="hist",
    learning_rate=0.05,
    max_depth=6,
    n_estimators=300,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    lambdarank_pair_method="topk",
    lambdarank_num_pair_per_sample=8
)

ranker.fit(
    X_train_sorted,
    y_train_sorted,
    qid=qid_train_sorted,
    eval_set=[(X_valid_sorted, y_valid_sorted)],
    eval_qid=[qid_valid_sorted],
    verbose=True
)

[0]	validation_0-ndcg@8:0.49766
[1]	validation_0-ndcg@8:0.54293
[2]	validation_0-ndcg@8:0.59521
[3]	validation_0-ndcg@8:0.63512
[4]	validation_0-ndcg@8:0.64603
[5]	validation_0-ndcg@8:0.65179
[6]	validation_0-ndcg@8:0.65429
[7]	validation_0-ndcg@8:0.65436
[8]	validation_0-ndcg@8:0.65756
[9]	validation_0-ndcg@8:0.65827
[10]	validation_0-ndcg@8:0.66102
[11]	validation_0-ndcg@8:0.66189
[12]	validation_0-ndcg@8:0.66314
[13]	validation_0-ndcg@8:0.66421
[14]	validation_0-ndcg@8:0.66308
[15]	validation_0-ndcg@8:0.66303
[16]	validation_0-ndcg@8:0.66316
[17]	validation_0-ndcg@8:0.66512
[18]	validation_0-ndcg@8:0.66593
[19]	validation_0-ndcg@8:0.66524
[20]	validation_0-ndcg@8:0.66525
[21]	validation_0-ndcg@8:0.66630
[22]	validation_0-ndcg@8:0.66619
[23]	validation_0-ndcg@8:0.66669
[24]	validation_0-ndcg@8:0.66600
[25]	validation_0-ndcg@8:0.66525
[26]	validation_0-ndcg@8:0.66422
[27]	validation_0-ndcg@8:0.66374
[28]	validation_0-ndcg@8:0.66417
[29]	validation_0-ndcg@8:0.66386
[30]	validation_0-nd

,objective,'rank:ndcg'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
train_df_sorted = train_df.copy()
valid_df_sorted = valid_df.copy()
test_df_sorted = test_df.copy()

# একই sorting apply
train_df_sorted = train_df_sorted.assign(_row_order=np.arange(len(train_df_sorted)))
valid_df_sorted = valid_df_sorted.assign(_row_order=np.arange(len(valid_df_sorted)))
test_df_sorted  = test_df_sorted.assign(_row_order=np.arange(len(test_df_sorted)))

train_df_sorted = train_df_sorted.sort_values(group_col).reset_index(drop=True)
valid_df_sorted = valid_df_sorted.sort_values(group_col).reset_index(drop=True)
test_df_sorted  = test_df_sorted.sort_values(group_col).reset_index(drop=True)

train_scores = ranker.predict(X_train_sorted)
valid_scores = ranker.predict(X_valid_sorted)
test_scores  = ranker.predict(X_test_sorted)

train_df_sorted["pred_score"] = train_scores
valid_df_sorted["pred_score"] = valid_scores
test_df_sorted["pred_score"] = test_scores

display(test_df_sorted.head())

,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id,_row_order,pred_score
0,23494,708,Spices-Cooking,6,14.166667,94.0,2.0,3.0,0.979264,Winter,0,0,Afternoon,1,1,27,0,0.053469
1,23494,15262,Meat-Fresh,6,30.500000,99.0,1.0,19.0,0.988786,Winter,0,0,Afternoon,1,0,27,1,-0.504754
2,23494,890,Spices-Cooking,6,12.333333,94.0,1.0,11.0,0.979264,Winter,0,0,Afternoon,1,0,27,2,-0.213063
3,23494,15104,Veg-Cooking,6,7.166667,47.0,3.0,2.0,0.987591,Winter,0,0,Afternoon,1,0,27,3,0.646124
4,23494,32441,Pantry-Oils,6,23.833333,54.0,5.0,9.0,0.995188,Winter,0,0,Afternoon,1,0,27,4,-0.212397


In [14]:
def precision_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1][:k]
    topk_true = np.array(y_true)[order]
    return float(np.sum(topk_true)) / float(k)


def recall_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true)
    total_relevant = np.sum(y_true)
    
    if total_relevant == 0:
        return 0.0
    
    order = np.argsort(y_score)[::-1][:k]
    topk_true = y_true[order]
    return float(np.sum(topk_true)) / float(total_relevant)


def ndcg_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true).reshape(1, -1)
    y_score = np.array(y_score).reshape(1, -1)
    return float(ndcg_score(y_true, y_score, k=k))

In [15]:
def evaluate_grouped_ranking(df_scored, group_col="group_id", label_col="label", score_col="pred_score", k=5):
    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    
    for gid, grp in df_scored.groupby(group_col):
        y_true = grp[label_col].values
        y_score = grp[score_col].values
        
        precision_scores.append(precision_at_k(y_true, y_score, k=k))
        recall_scores.append(recall_at_k(y_true, y_score, k=k))
        ndcg_scores.append(ndcg_at_k(y_true, y_score, k=k))
    
    return {
        f"Precision@{k}": float(np.mean(precision_scores)),
        f"Recall@{k}": float(np.mean(recall_scores)),
        f"NDCG@{k}": float(np.mean(ndcg_scores))
    }

train_metrics_k5 = evaluate_grouped_ranking(train_df_sorted, k=5)
valid_metrics_k5 = evaluate_grouped_ranking(valid_df_sorted, k=5)
test_metrics_k5  = evaluate_grouped_ranking(test_df_sorted, k=5)

print("Train metrics @5:", train_metrics_k5)
print("Valid metrics @5:", valid_metrics_k5)
print("Test metrics @5 :", test_metrics_k5)

Train metrics @5: {'Precision@5': 0.18412426614481414, 'Recall@5': 0.9206213307240705, 'NDCG@5': 0.7307353040053662}
Valid metrics @5: {'Precision@5': 0.18087084148727986, 'Recall@5': 0.9043542074363993, 'NDCG@5': 0.700923186921953}
Test metrics @5 : {'Precision@5': 0.18082191780821918, 'Recall@5': 0.9041095890410958, 'NDCG@5': 0.7040131883750427}


In [16]:
for k in [3, 5, 10]:
    print(f"\n===== Metrics at K = {k} =====")
    print("Train:", evaluate_grouped_ranking(train_df_sorted, k=k))
    print("Valid:", evaluate_grouped_ranking(valid_df_sorted, k=k))
    print("Test :", evaluate_grouped_ranking(test_df_sorted, k=k))


===== Metrics at K = 3 =====
Train: {'Precision@3': 0.2668684768427919, 'Recall@3': 0.8006054305283757, 'NDCG@3': 0.6811865062520801}
Valid: {'Precision@3': 0.25799086757990863, 'Recall@3': 0.773972602739726, 'NDCG@3': 0.647283823821032}
Test : {'Precision@3': 0.2576647097195042, 'Recall@3': 0.7729941291585127, 'NDCG@3': 0.6499890152008546}

===== Metrics at K = 5 =====
Train: {'Precision@5': 0.18412426614481414, 'Recall@5': 0.9206213307240705, 'NDCG@5': 0.7307353040053662}
Valid: {'Precision@5': 0.18087084148727986, 'Recall@5': 0.9043542074363993, 'NDCG@5': 0.700923186921953}
Test : {'Precision@5': 0.18082191780821918, 'Recall@5': 0.9041095890410958, 'NDCG@5': 0.7040131883750427}

===== Metrics at K = 10 =====
Train: {'Precision@10': 0.0998624021526419, 'Recall@10': 0.9986240215264188, 'NDCG@10': 0.7568130934628561}
Valid: {'Precision@10': 0.09982876712328767, 'Recall@10': 0.9982876712328768, 'NDCG@10': 0.7322912082222018}
Test : {'Precision@10': 0.09985322896281802, 'Recall@10': 0.9

In [17]:
feature_importance_df = pd.DataFrame({
    "feature": X_train_sorted.columns,
    "importance": ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(20))

,feature,importance
48,candidate_category_Personal-Care-Bath,0.047803
49,candidate_category_Personal-Care-Cosmetics,0.042253
38,candidate_category_Meat-Fresh,0.040890
50,candidate_category_Personal-Care-Hair,0.039157
44,candidate_category_Pantry-Oils,0.032457
37,candidate_category_Instant-Food,0.031131
30,candidate_category_Fruits-Fresh,0.030532
19,candidate_category_Beverage-Juice,0.030503
26,candidate_category_Dairy-Other,0.029533
54,candidate_category_Protein-Egg,0.029393


In [18]:
MODEL_FILE = MODEL_DIR / "xgboost_ranker_model.json"
FEATURE_FILE = MODEL_DIR / "ranker_feature_columns.json"
IMPORTANCE_FILE = MODEL_DIR / "ranker_feature_importance.csv"

ranker.save_model(MODEL_FILE)

with open(FEATURE_FILE, "w", encoding="utf-8") as f:
    json.dump(list(X_train_sorted.columns), f, ensure_ascii=False, indent=2)

feature_importance_df.to_csv(IMPORTANCE_FILE, index=False)

print("Saved model:", MODEL_FILE)
print("Saved feature list:", FEATURE_FILE)
print("Saved importance:", IMPORTANCE_FILE)

Saved model: D:\recommendation_item_API\data\model_outputs\xgboost_ranker_model.json
Saved feature list: D:\recommendation_item_API\data\model_outputs\ranker_feature_columns.json
Saved importance: D:\recommendation_item_API\data\model_outputs\ranker_feature_importance.csv


In [19]:
sample_group = test_df_sorted[group_col].iloc[0]
sample_view = test_df_sorted[test_df_sorted[group_col] == sample_group].copy()

sample_view = sample_view.sort_values("pred_score", ascending=False)

display(
    sample_view[
        [
            "group_id",
            "candidate_item_id",
            "candidate_category",
            "label",
            "pred_score",
            "basket_size",
            "item_cooccurrence_score",
            "category_affinity_score",
            "context_popularity_score",
            "customer_history_score",
            "embedding_similarity_score"
        ]
    ]
)

,group_id,candidate_item_id,candidate_category,label,pred_score,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score
3,27,15104,Veg-Cooking,0,0.646124,6,7.166667,47.0,3.0,2.0,0.987591
0,27,708,Spices-Cooking,1,0.053469,6,14.166667,94.0,2.0,3.0,0.979264
5,27,13917,Fish-Fresh,0,0.031360,6,14.166667,106.0,2.0,35.0,0.988419
4,27,32441,Pantry-Oils,0,-0.212397,6,23.833333,54.0,5.0,9.0,0.995188
2,27,890,Spices-Cooking,0,-0.213063,6,12.333333,94.0,1.0,11.0,0.979264
7,27,6815,Beverage-Hot,0,-0.277879,6,5.666667,16.0,4.0,2.0,0.998512
8,27,14978,Meat-Fresh,0,-0.336690,6,29.333333,99.0,3.0,9.0,0.988786
1,27,15262,Meat-Fresh,0,-0.504754,6,30.500000,99.0,1.0,19.0,0.988786
6,27,2038,Pantry-Oils,0,-0.543869,6,27.333333,54.0,2.0,5.0,0.995188
10,27,25474,Pantry-Dryfood,0,-3.621063,6,29.333333,8.0,0.0,6.0,0.992151
